# Парсер данных Telegram

Проект создан командой полной свэга SWAG

Руководитель команды - Новиков Александр Константинович

Участники команды - Крюков Алексей Алексеевич, Жыргалбек кызы Жасмин

Студенты 1-го курса МП "Социология публичной сферы и цифровая аналитика", трек "Цифровая аналитика социальных процессов"


Часть 1. Конфигурация и подключение к Telegram API

In [ ]:
from telethon import TelegramClient
import csv
import os
import json
from typing import Optional, Dict, Any, List, Set

API_ID = "NUMBERS HERE" # числовой идентификатор приложения Telegram
API_HASH = "HASH HERE" # строковый хэш приложения Telegram
SESSION_NAME = "NAME HERE" # имя файла сессии

OUTPUT_DIR = "NAME HERE" # название файла, куда сохраняются результаты
POSTS_LIMIT = 200  # максимальное количество сообщений для каждого источника (чтобы не перегружать систему)
DOWNLOAD_MEDIA = False  # загрузка медиафайлов не используется

Часть 2. Вспомогательные функции обработки данных Telegram

In [ ]:
def ensure_dir(path: str) -> None:
    """Создаёт папку, если она ещё не существует."""
    if not os.path.exists(path):
        os.makedirs(path)

def safe_text(value: Optional[str]) -> str:
    """Возвращает строку или пустую строку, если значение отсутствует."""
    return value if value is not None else ""

def detect_chat_type(entity) -> str:
    """
    Определяет тип сущности в формате из схемы:
    channel / group / supergroup.
    """
    if getattr(entity, "broadcast", False):
        return "channel"
    if getattr(entity, "megagroup", False):
        return "supergroup"
    return "group"

def display_name_from_sender(sender) -> str:
    """Формирует отображаемое имя отправителя."""
    if sender is None:
        return "неизвестный_пользователь"

    if hasattr(sender, "first_name") or hasattr(sender, "last_name"):
        first = getattr(sender, "first_name", "") or ""
        last = getattr(sender, "last_name", "") or ""
        full_name = f"{first} {last}".strip()
        return full_name if full_name else "неизвестный_пользователь"

    if hasattr(sender, "title"):
        return sender.title or "неизвестный_источник"

    return "неизвестный_пользователь"

def total_reaction_count(msg) -> int:
    """Возвращает суммарное количество реакций на сообщение."""
    if not getattr(msg, "reactions", None):
        return 0

    results = getattr(msg.reactions, "results", None)
    if not results:
        return 0

    return sum(getattr(item, "count", 0) or 0 for item in results)

def extract_reactions(msg, entity_type: str, owner_message_id: int, owner_chat_id: int):
    """
    Возвращает список реакций в формате для таблицы tg_reactions.

    entity_type:
        post / comment

    owner_message_id:
        tg_message_id поста или комментария

    owner_chat_id:
        tg_chat_id чата или канала, в котором находится сообщение
    """
    rows = []

    if not getattr(msg, "reactions", None):
        return rows

    results = getattr(msg.reactions, "results", None)
    if not results:
        return rows

    for reaction_obj in results:
        reaction_repr = "неизвестная_реакция"
        reaction_value = getattr(reaction_obj, "reaction", None)

        if reaction_value is not None:
            emoticon = getattr(reaction_value, "emoticon", None)
            if emoticon:
                reaction_repr = emoticon
            else:
                reaction_repr = str(reaction_value)

        rows.append({
            "entity_type": entity_type,
            "owner_tg_chat_id": owner_chat_id,
            "owner_tg_message_id": owner_message_id,
            "reaction": reaction_repr,
            "reaction_count": getattr(reaction_obj, "count", 0) or 0
        })

    return rows

async def get_input_entity(client: TelegramClient, chat_input: str):
    """
    Принимает username, ссылку или числовой идентификатор
    и возвращает сущность Telegram.
    """
    raw = chat_input.strip()

    if raw.startswith("@"):
        raw = raw[1:]

    try:
        raw = int(raw)
    except ValueError:
        pass

    return await client.get_entity(raw)

def write_csv(filename: str, rows: List[Dict[str, Any]], fieldnames: List[str]) -> None:
    """Сохраняет список словарей в CSV-файл."""
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

In [ ]:
async def export_telegram_data(client: TelegramClient, chat_inputs: List[str]):
    """
    Собирает данные из нескольких чатов, каналов или групп
    и сохраняет общий результат в CSV-файлы.

    chat_inputs:
        Список username, ссылок или идентификаторов источников.
    """
    ensure_dir(OUTPUT_DIR)

    chats_rows: List[Dict[str, Any]] = []
    users_map: Dict[int, Dict[str, Any]] = {}
    posts_rows: List[Dict[str, Any]] = []
    comments_rows: List[Dict[str, Any]] = []
    reactions_rows: List[Dict[str, Any]] = []

    processed_chat_ids: Set[int] = set()
    processed_post_keys: Set[tuple] = set()
    processed_comment_keys: Set[tuple] = set()
    processed_reaction_keys: Set[tuple] = set()

    summary_rows: List[Dict[str, Any]] = []

    for chat_input in chat_inputs:
        try:
            entity = await get_input_entity(client, chat_input)
            full_entity = await client.get_entity(entity)
        except Exception as error:
            print(f"Не удалось получить доступ к источнику '{chat_input}': {error}")
            continue

        chat_type = detect_chat_type(full_entity)
        tg_chat_id = getattr(full_entity, "id", None)
        title = getattr(full_entity, "title", None) or str(chat_input)
        invite_link = None

        if tg_chat_id not in processed_chat_ids:
            chats_rows.append({
                "tg_chat_id": tg_chat_id,
                "title": title,
                "type": chat_type,
                "invite_link": invite_link
            })
            processed_chat_ids.add(tg_chat_id)

        print(f"Собираем данные из источника: {title} ({chat_type})")

        messages_by_id: Dict[int, Any] = {}
        root_post_map: Dict[int, int] = {}

        async for msg in client.iter_messages(full_entity, reverse=True, limit=POSTS_LIMIT):
            if msg is None:
                continue
            messages_by_id[msg.id] = msg

        sorted_message_ids = sorted(messages_by_id.keys())

        for msg_id in sorted_message_ids:
            msg = messages_by_id[msg_id]
            reply_to_msg_id = getattr(msg, "reply_to_msg_id", None)

            if reply_to_msg_id is None:
                root_post_map[msg_id] = msg_id
            else:
                parent_root = root_post_map.get(reply_to_msg_id)
                if parent_root is not None:
                    root_post_map[msg_id] = parent_root

        posts_count_before = len(posts_rows)
        comments_count_before = len(comments_rows)
        reactions_count_before = len(reactions_rows)

        for msg_id in sorted_message_ids:
            msg = messages_by_id[msg_id]

            sender = None
            sender_id = getattr(msg, "sender_id", None)

            try:
                sender = await msg.get_sender()
            except Exception:
                sender = None

            if sender_id is not None and sender_id not in users_map:
                users_map[sender_id] = {
                    "tg_user_id": sender_id,
                    "display_name": display_name_from_sender(sender),
                    "username": safe_text(getattr(sender, "username", None)) if sender else ""
                }

            text = msg.message or ""
            msg_reaction_count = total_reaction_count(msg)
            root_post_id = root_post_map.get(msg.id)

            is_post = root_post_id == msg.id

            if is_post:
                comment_count = 0
                replies_obj = getattr(msg, "replies", None)
                if replies_obj is not None:
                    comment_count = getattr(replies_obj, "replies", 0) or 0

                post_key = (tg_chat_id, msg.id)
                if post_key not in processed_post_keys:
                    posts_rows.append({
                        "tg_message_id": msg.id,
                        "tg_chat_id": tg_chat_id,
                        "author_tg_user_id": sender_id,
                        "published_at": msg.date.isoformat() if msg.date else "",
                        "text": text,
                        "view_count": getattr(msg, "views", None),
                        "reaction_count": msg_reaction_count,
                        "comment_count": comment_count
                    })
                    processed_post_keys.add(post_key)

                reaction_rows = extract_reactions(
                    msg,
                    entity_type="post",
                    owner_message_id=msg.id,
                    owner_chat_id=tg_chat_id
                )

                for reaction_row in reaction_rows:
                    reaction_key = (
                        reaction_row["entity_type"],
                        reaction_row["owner_tg_chat_id"],
                        reaction_row["owner_tg_message_id"],
                        reaction_row["reaction"]
                    )
                    if reaction_key not in processed_reaction_keys:
                        reactions_rows.append(reaction_row)
                        processed_reaction_keys.add(reaction_key)

            else:
                if root_post_id is None:
                    continue

                comment_key = (tg_chat_id, msg.id)
                if comment_key not in processed_comment_keys:
                    comments_rows.append({
                        "tg_message_id": msg.id,
                        "tg_chat_id": tg_chat_id,
                        "post_tg_message_id": root_post_id,
                        "author_tg_user_id": sender_id,
                        "published_at": msg.date.isoformat() if msg.date else "",
                        "text": text,
                        "reaction_count": msg_reaction_count
                    })
                    processed_comment_keys.add(comment_key)

                reaction_rows = extract_reactions(
                    msg,
                    entity_type="comment",
                    owner_message_id=msg.id,
                    owner_chat_id=tg_chat_id
                )

                for reaction_row in reaction_rows:
                    reaction_key = (
                        reaction_row["entity_type"],
                        reaction_row["owner_tg_chat_id"],
                        reaction_row["owner_tg_message_id"],
                        reaction_row["reaction"]
                    )
                    if reaction_key not in processed_reaction_keys:
                        reactions_rows.append(reaction_row)
                        processed_reaction_keys.add(reaction_key)

        summary_rows.append({
            "chat_title": title,
            "chat_type": chat_type,
            "tg_chat_id": tg_chat_id,
            "posts_count": len(posts_rows) - posts_count_before,
            "comments_count": len(comments_rows) - comments_count_before,
            "reactions_count": len(reactions_rows) - reactions_count_before
        })

    users_rows = list(users_map.values())

    write_csv(
        "tg_chats.csv",
        chats_rows,
        ["tg_chat_id", "title", "type", "invite_link"]
    )

    write_csv(
        "tg_users.csv",
        users_rows,
        ["tg_user_id", "display_name", "username"]
    )

    write_csv(
        "tg_posts.csv",
        posts_rows,
        [
            "tg_message_id",
            "tg_chat_id",
            "author_tg_user_id",
            "published_at",
            "text",
            "view_count",
            "reaction_count",
            "comment_count"
        ]
    )

    write_csv(
        "tg_comments.csv",
        comments_rows,
        [
            "tg_message_id",
            "tg_chat_id",
            "post_tg_message_id",
            "author_tg_user_id",
            "published_at",
            "text",
            "reaction_count"
        ]
    )

    write_csv(
        "tg_reactions.csv",
        reactions_rows,
        [
            "entity_type",
            "owner_tg_chat_id",
            "owner_tg_message_id",
            "reaction",
            "reaction_count"
        ]
    )

    with open(os.path.join(OUTPUT_DIR, "summary.json"), "w", encoding="utf-8") as file:
        json.dump({
            "sources_count": len(chats_rows),
            "users_count": len(users_rows),
            "posts_count": len(posts_rows),
            "comments_count": len(comments_rows),
            "reactions_count": len(reactions_rows),
            "sources": summary_rows
        }, file, ensure_ascii=False, indent=2)

    print("Сбор данных завершён.")
    print(f"Папка экспорта: {OUTPUT_DIR}")
    print(f"Количество источников: {len(chats_rows)}")
    print(f"Количество пользователей: {len(users_rows)}")
    print(f"Количество постов: {len(posts_rows)}")
    print(f"Количество комментариев: {len(comments_rows)}")
    print(f"Количество реакций: {len(reactions_rows)}")

async def main():
    sources: List[str] = []

    print("Введите username, ссылку или идентификатор чата, канала или группы.")
    print("Добавляйте по одному источнику в строке.")
    print("Когда закончите ввод, напишите STOP.")

    while True:
        chat_input = input("Для остановки введите STOP. Источник: ").strip()

        if not chat_input:
            print("Пустая строка пропущена.")
            continue

        if chat_input.upper() == "STOP":
            break

        sources.append(chat_input)

    if not sources:
        print("Список источников пуст. Сбор данных не выполнен.")
        return

    async with TelegramClient(SESSION_NAME, API_ID, API_HASH) as client:
        await export_telegram_data(client, sources)

await main()